# Notebook 01: Data Exploration
## River Water Mask Study — SAR vs. Optical

**Researcher:** Bouchra Daddaoui  
**Supervisors:** Dr. Michael Nones, Dr. Kaveh Ghahraman  
**Institute:** Institute of Geophysics, Polish Academy of Sciences

---

### Objectives
1. Visualize selected rivers on a global map
2. Explore available Sentinel-1 and Sentinel-2 scenes per river
3. Assess temporal data coverage (cloud cover, SAR pass frequency)
4. Visualize sample RGB composites and SAR backscatter images
5. Load and inspect JRC reference water masks


In [ ]:
# ─── Setup ───────────────────────────────────────────────
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.data_loader import RiverDataLoader, get_selected_rivers, RIVER_CONFIG

# Initialize GEE (requires authentication: ee.Authenticate() once)
ee.Initialize()
print('GEE initialized successfully.')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Study River Overview

In [ ]:
# Load river selection table
rivers_df = get_selected_rivers()
display(rivers_df[['river_id', 'river_name', 'country', 'lat_center', 'lon_center',
                    'altitude_m', 'mean_discharge_m3s', 'climate_zone', 'annual_cloud_cover_pct']])

In [ ]:
# Global map of selected rivers
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.GeoDataFrame(
    rivers_df,
    geometry=[Point(lon, lat) for lon, lat in
              zip(rivers_df['lon_center'], rivers_df['lat_center'])],
    crs='EPSG:4326'
)

world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

fig, ax = plt.subplots(figsize=(16, 8))
world.plot(ax=ax, color='#ECEFF1', edgecolor='#B0BEC5', linewidth=0.5)
gdf.plot(ax=ax, color='#1565C0', markersize=80, zorder=5)

for _, row in rivers_df.iterrows():
    ax.annotate(row['river_name'], (row['lon_center'], row['lat_center']),
                xytext=(5, 5), textcoords='offset points', fontsize=10,
                fontweight='bold', color='#1565C0')

ax.set_title('Study River Locations (GLORIN-based Selection)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('../results/figures/study_rivers_map.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Data Availability Assessment

In [ ]:
# Check Sentinel-1 and Sentinel-2 scene counts per river
results = []

for river_id in rivers_df['river_id']:
    loader = RiverDataLoader(river_id, '2020-01-01', '2022-12-31')
    
    s1_count = loader.get_sentinel1().size().getInfo()
    s2_count = loader.get_sentinel2(max_cloud=100, apply_cloud_mask=False).size().getInfo()
    s2_clear = loader.get_sentinel2(max_cloud=20).size().getInfo()
    jrc_count = loader.get_jrc_reference().size().getInfo()
    
    results.append({
        'river_id': river_id,
        'river': RIVER_CONFIG[river_id]['name'],
        'S1_scenes': s1_count,
        'S2_total': s2_count,
        'S2_clear_sky': s2_clear,
        'pct_clear': round(s2_clear / s2_count * 100, 1) if s2_count > 0 else 0,
        'JRC_months': jrc_count,
    })
    print(f'{river_id}: S1={s1_count}, S2={s2_count} (clear={s2_clear})')

availability_df = pd.DataFrame(results)
display(availability_df)

In [ ]:
# Plot data availability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# S1 vs S2 scene count
x = np.arange(len(availability_df))
axes[0].bar(x - 0.2, availability_df['S1_scenes'], 0.35, label='Sentinel-1', color='#9C27B0')
axes[0].bar(x + 0.2, availability_df['S2_clear_sky'], 0.35, label='S2 Clear (<20% cloud)', color='#4CAF50')
axes[0].set_xticks(x)
axes[0].set_xticklabels(availability_df['river'], rotation=30, ha='right')
axes[0].set_ylabel('Number of Scenes (2020–2022)')
axes[0].set_title('Satellite Scene Availability per River')
axes[0].legend()

# Clear sky percentage
axes[1].barh(availability_df['river'], availability_df['pct_clear'],
             color=['#E53935' if p < 30 else '#FB8C00' if p < 60 else '#4CAF50'
                    for p in availability_df['pct_clear']])
axes[1].axvline(30, color='red', linestyle='--', alpha=0.5, label='30% threshold')
axes[1].set_xlabel('% Scenes with <20% Cloud Cover')
axes[1].set_title('Cloud-Free Optical Data Availability')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/data_availability.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Sample Image Visualization

In [ ]:
# Interactive GEE Map — visualize one river
RIVER_TO_EXPLORE = 'R06'  # Change to explore different rivers

loader = RiverDataLoader(RIVER_TO_EXPLORE, '2021-06-01', '2021-08-31')
aoi = loader.get_aoi()

# Get best optical image
s2 = loader.get_sentinel2(max_cloud=20).first()

# Get SAR image (median composite)
s1 = loader.get_sentinel1().median()

# Get JRC reference
jrc = loader.get_jrc_reference().select('water').max()

# Create interactive map
Map = geemap.Map(center=[RIVER_CONFIG[RIVER_TO_EXPLORE]['lat'],
                          RIVER_CONFIG[RIVER_TO_EXPLORE]['lon']], zoom=10)

Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Sentinel-2 RGB')
Map.addLayer(s1.select('VV'), {'min': -25, 'max': 0, 'palette': ['black', 'white']}, 'S1 VV (dB)')
Map.addLayer(jrc.eq(2), {'palette': ['blue']}, 'JRC Water Reference')
Map.addLayer(aoi, {}, 'AOI')

Map

## 4. Seasonal Backscatter Profiles

In [ ]:
# Plot monthly mean VV backscatter for one river
river_id = 'R04'  # Ganges — monsoon-driven
loader = RiverDataLoader(river_id, '2020-01-01', '2022-12-31')

# NOTE: Replace with actual GEE time series extraction
# This is a placeholder for the expected output format
months = pd.date_range('2020-01', '2022-12', freq='M')
# mean_vv_db = [actual values from GEE .getInfo()]

print(f'Analyzing seasonal SAR profile for {RIVER_CONFIG[river_id]["name"]} river...')
print('(Run GEE export to get actual time series values)')

## Summary

Key findings from data exploration:
- [ ] Document S1/S2 availability per river
- [ ] Identify rivers with critical cloud cover gaps
- [ ] Note any data gaps in JRC reference
- [ ] Confirm AOI coverage is adequate for river width sampling
- [ ] Flag any data quality issues (striping, noise, orbit gaps)